# arc3-duck-nvfp4-avo - NVFP4 serving stack + the AVO solver

**This is a reproduction of other people's work, run on our own account to
measure it. No score quoted by any upstream is ours.** Full credit and licence
text in `THIRD_PARTY_NOTICE.md`.

Identical to `calamitychasm/arc3-duck-nvfp4-anim` except that the **solver
bundle** is swapped. The serving stack, the model, the analyzer knobs
(`LOCAL_ANALYZER_SEED=20260825`, `LOCAL_ANALYZER_YIELD_SECONDS=180`),
`concurrency=28`, `max_runtime_s_per_game=7920` and the run loop are all
carried over byte-identically, and the build script asserts every one of the
18 inherited cells by sha256.

Upstreams:

- **Solver** - the Tufa Labs Duck harness, on Jakob Brueggen's
  `experiment/avo-v2` branch (`ARC3-Inference 74ff3df`), mounted as the frozen
  pin **`raist321/taaf-avo-v27-bundle`** rather than the rolling
  `jakobbrggen/taaf-kaggle-source` slug, and executed unmodified. Verified
  byte-identical to that slug's v27 on 2026-09-19.
- **Serving / weights** - Keith Tyser's `duck-qwen3-8-flash-next-nvfp4-mtp`
  appliance over RadixArk's NVFP4 quantisation of Qwen3.8-Flash-Next.
  Unmodified and sealed.
- **The NVFP4+anim graft this is patched onto** - Thuitanium / Knowless Crew
  (`yocybercode/thui-animfast-b71-full25-r1`).
- **AVO** - Tufa Labs' reimplementation, from the published description, of the
  architecture NVIDIA's AVO team described. NVIDIA released no code and no
  ablations, and their 100.00 public-set figure was obtained with a frontier
  model, not this one.

## What AVO adds

`inference/avo/`, 733 lines, subclassing `ToolAgent` rather than replacing it:

1. **Durable memory** - the base agent clears `_summarized_knowledge` whenever
   the runtime dir changes, i.e. at every level boundary, which is exactly
   where it is worth most. AVO persists it and reloads it per game.
2. **Stagnation supervisor** - two triggers (`barren_turns >= 3`, which yields
   to frame novelty, and `unrewarded_turns >= 12`, which ignores it), with a
   3-step escalation to a hard redirect. **Zero extra model calls**: it appends
   a paragraph to the prompt the turn already sends.
3. **Exploit deadline** - at 0.6 x 7,920 s = 4,752 s the arm stops building a
   world model and plays its best policy.
4. **Phased loop** - **disabled in this arm** (`ARC3_AVO_PHASED_LOOP=0`, the bundle's own
   ablation switch). Its `INSPECT` phase opens *"Do not act this turn unless
   the situation is already unambiguous"*, which would instruct one turn in four
   not to act on top of a measured 37.7% dead-turn rate. That is a second
   variable and it is held off here.

## Why the score of this free run cannot rank the arm

A single public-25 mean carries SE +/-2.46 (`experiments/stage7_noise_floor.md`).
This run is for **counted** telemetry -- dead-turn rate, actions, calls, levels
by index, supervisor firings -- and for catastrophe detection. The incumbent's
matched free run is **9.97 mean / 2,615 actions / 42 levels / 20-of-25 games
scoring**; the mean is reported only for comparability, not to rank anything.


In [ ]:
# [calamitychasm] ADDED FOR THIS FORK -- diagnostic only, changes no behaviour.
# The upstream README warns that a manual copy must select the RTX PRO 6000 by hand.
# We push via the API with machine_shape=NvidiaRtxPro6000, which IS honoured (verified
# in experiments/stage7_duck_nvfp4.md), but a wrong card would waste the whole run, so
# print what we actually got before anything expensive happens.
import os, shutil, subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "nvidia-smi unavailable")
try:
    _mem_kb = int(next(l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")))
    _ram = f"{_mem_kb / 1048576:.1f}"
except Exception:
    _ram = "?"
print(f"HW_PROBE host_ram_gib={_ram}  cpu_count={os.cpu_count()}  "
      f"free_disk_gib={shutil.disk_usage('/kaggle/working').free / 2**30:.1f}  "
      f"rerun={os.getenv('KAGGLE_IS_COMPETITION_RERUN')!r}")


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
# thui-animfast: full diagnostics on an interactive public run (usage/events/transcript sidecars); minimal in a rerun.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# thui-animfast: resolve the competition mount instead of assuming its layout -- Kaggle serves either
# /kaggle/input/competitions/<comp> or /kaggle/input/<comp>, and which one varies between runs.
_COMP_CANDIDATES = ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3"]
_COMP_DIR = next((_p for _p in _COMP_CANDIDATES if os.path.isdir(_p)), None)
assert _COMP_DIR is not None, (
    "thui-animfast: no competition mount found. Tried " + repr(_COMP_CANDIDATES)
    + "; /kaggle/input holds "
    + repr(sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING")
)
_WHEELS = os.path.join(_COMP_DIR, "arc_agi_3_wheels")
assert os.path.isdir(_WHEELS), "thui-animfast: resolved wheels dir is not a directory: " + _WHEELS
print("thui-animfast: competition mount = " + _COMP_DIR, flush=True)
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _WHEELS,
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1", "raist321/taaf-avo-v27-bundle"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir(label: str) -> Path:
    # thui-animfast: TWO attached datasets carry the marker (his June duck bundle and the anim bundle), so
    # "first marker wins" is a coin flip -- pick by the benchmark_label the marker file records.
    found = {}
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        try:
            found[json.loads(marker.read_text())["benchmark_label"]] = marker.parent
        except Exception as exc:
            print(f"thui-animfast: unreadable marker {marker}: {exc!r}", flush=True)
    if label not in found:
        raise RuntimeError(f"TAAF source bundle {label!r} not found under /kaggle/input; markers = {found}")
    return found[label]


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir("duck-harness-kaggle")          # his: serving_setup.py, vllm patches, watchdog, teardown
ANIM_BUNDLE_DIR = _find_bundle_dir("avo-kaggle")     # [calamitychasm] ours: the AVO solver tree (anim + inference/avo) + its pickles.
# Name kept so cell 9's graft teeth stay byte-identical. The inherited animation.py
# assert two lines below is what proves this bundle is still the anim superset.
assert BUNDLE_DIR != ANIM_BUNDLE_DIR, "thui-animfast: both labels resolved to one directory"
assert (BUNDLE_DIR / "serving_setup.py").is_file(), f"thui-animfast: his bundle has no serving_setup.py: {BUNDLE_DIR}"
assert (ANIM_BUNDLE_DIR / "src" / "ARC3-Inference" / "inference" / "utils" / "animation.py").is_file(), (
    f"thui-animfast: the anim bundle has no animation.py: {ANIM_BUNDLE_DIR}")
print(f"thui-animfast: anim bundle = {ANIM_BUNDLE_DIR}", flush=True)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
# thui-animfast: his tree minus the two solver repos (the June duck), plus the anim solver tree. The loop below
# inserts each entry at sys.path[0], so the LAST entries win -- the anim ones; the .pth is written anim-first.
_SOLVER_REPOS = {"ARC3-Inference", "tufa-arc-agi-framework"}
_his_entries = [e for e in _source_path_entries(BUNDLE_DIR) if e.parent.name not in _SOLVER_REPOS and e.name not in _SOLVER_REPOS]
_anim_entries = _source_path_entries(ANIM_BUNDLE_DIR)
assert _anim_entries and all(str(e).startswith(str(ANIM_BUNDLE_DIR)) for e in _anim_entries), _anim_entries
assert not any(("ARC3-Inference" in str(e) or "tufa-arc-agi-framework" in str(e)) for e in _his_entries), _his_entries
source_entries = _his_entries + _anim_entries
print(f"thui-animfast: source roots his={[str(e) for e in _his_entries]} anim={[str(e) for e in _anim_entries]}", flush=True)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in (_anim_entries + _his_entries)))   # anim first for child processes
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# ---- thui-animfast: the thui-v3 knobs, set AFTER his serving_setup persisted the analyzer env and BEFORE any
# `inference` import (tool_agent reads LOCAL_ANALYZER_SEED / YIELD_SECONDS at import time), then the graft teeth.
_KNOBS = {"LOCAL_ANALYZER_SEED": "20260825", "LOCAL_ANALYZER_YIELD_SECONDS": "180"}
_persisted = json.loads(SETUP_ENV_PATH.read_text())
assert _persisted.get("LOCAL_ANALYZER_MODEL_ID") == "Qwen/Qwen3.8-Flash-Next-NVFP4", _persisted.get("LOCAL_ANALYZER_MODEL_ID")
assert _persisted.get("LOCAL_ANALYZER_YIELD_SECONDS") == "60", "his serving_setup no longer persists yield 60 -- re-derive the override"
assert _persisted.get("LOCAL_ANALYZER_TEMPERATURE") == "0.6" and _persisted.get("MULTIMODAL_UPSCALE") == "4", _persisted
_persisted.update(_KNOBS)
SETUP_ENV_PATH.write_text(json.dumps(_persisted, indent=2, sort_keys=True) + "\n")
os.environ.update(_KNOBS)
assert "inference" not in sys.modules and "taaf" not in sys.modules, "solver imported before the knob override"
import inference.agent.tool_agent as _tool_agent
import inference.utils.animation as _anim_mod
import taaf as _taaf
for _m in (_tool_agent, _anim_mod, _taaf):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)
assert _tool_agent._LOCAL_ANALYZER_SEED == int("20260825"), _tool_agent._LOCAL_ANALYZER_SEED
assert float(_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS) == float("180"), _tool_agent._LOCAL_ANALYZER_YIELD_SECONDS
assert os.environ["LOCAL_ANALYZER_MODEL_ID"] == "Qwen/Qwen3.8-Flash-Next-NVFP4"
print(f"THUI_ANIMFAST_GRAFT ok solver={Path(_tool_agent.__file__).parent} seed={_tool_agent._LOCAL_ANALYZER_SEED} "
      f"yield={_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS} model={os.environ['LOCAL_ANALYZER_MODEL_ID']} "
      f"temperature={os.environ['LOCAL_ANALYZER_TEMPERATURE']} upscale={os.environ['MULTIMODAL_UPSCALE']}", flush=True)


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(ANIM_BUNDLE_DIR / "deploy_target.pkl", "rb") as file:   # [calamitychasm] the AVO bundle's target -- 54000 s, overridden to 32400 in cell 13
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(ANIM_BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:   # thui-animfast: the anim solver
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR
# thui-animfast: the unpickled solver must be the anim chassis, not the June duck.
# [calamitychasm] AVO is that chassis plus inference/avo, so both halves are checked.
assert bm.label == "avo-kaggle", bm.label
assert getattr(bm.solver, "animation_awareness", None) is True and getattr(bm.solver, "hard_noop_guard", None) is True, vars(bm.solver)
assert getattr(bm.solver, "avo_agent", None) is True, vars(bm.solver)   # [calamitychasm] the point of this arm
assert type(bm.solver).__module__ == "inference.framework.solver"
print(f"thui-animfast: bm.label={bm.label} solver={type(bm.solver).__name__} animation_awareness={bm.solver.animation_awareness} avo_agent={bm.solver.avo_agent} "
      f"hard_noop_guard={bm.solver.hard_noop_guard} target.max_runtime_s={target.max_runtime_s}", flush=True)


## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts â€” the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Exact public-25 and competition settings.
bm.solver.max_runtime_s_per_game = 7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 28
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
# [calamitychasm] AVO TRAP 1 -- SET the notebook budget, do not assert it.
# The anim bundle's deploy_target.pkl carried 32400.0 and the inherited cell asserted it.
# The AVO bundle's carries 54000.0 (15 h -- inference/avo/settings.py's "15h Kaggle run").
# Kaggle's hard cap is 9 h, so asserting here fails the run and inheriting 54000 overruns
# it. Set 32400 and print the override so it is visible in the log instead of silent.
# Still fails loudly on a third value: that would mean the pinned bundle moved.
_inherited_max_runtime_s = float(getattr(target, 'max_runtime_s', 0.0) or 0.0)
if _inherited_max_runtime_s not in (32400.0, 54000.0):
    raise RuntimeError(
        f'Unexpected inherited notebook budget {target.max_runtime_s!r}; expected 32400 (anim) '
        f'or 54000 (avo-v27). The pinned bundle moved -- re-review before running.'
    )
target.max_runtime_s = 32400.0
print(
    f'AVO_BUDGET_OVERRIDE inherited_s={_inherited_max_runtime_s} applied_s={target.max_runtime_s} '
    f'per_game_s={bm.solver.max_runtime_s_per_game} '
    f'exploit_deadline_s={bm.solver.max_runtime_s_per_game * 0.6}',
    flush=True,
)
print(
    f'PUBLIC25_SETTINGS budget_s={bm.solver.max_runtime_s_per_game} '
    f'concurrency={bm.solver.concurrency} analyzer_timeout={bm.solver.analyzer_timeout} '
    f'action_cap={bm.solver.max_actions_per_game} request_logs={bm.solver.save_request_logs}',
    flush=True,
)


## 6b. The AVO arm

The one behavioural variable in this notebook relative to the arm it will be
compared against. `ARC3_AVO_PHASED_LOOP` is the bundle's own ablation switch:
off gives memory + stagnation supervisor + exploit deadline against the same
base agent; on adds the four-phase inspect/plan/implement/evaluate directive.

The flag is *verified* here, not merely exported: `AvoSettings.from_env()` is
the exact call the solver makes per game, so reading it back is the same value
the run will use. It also resolves `inference.avo` off the mounted bundle, which
is the check that catches a stale `sys.path` before 9 hours are spent on it.


In [ ]:
# [calamitychasm] THE ONE VARIABLE OF THIS ARM.
# memory + supervisor + exploit deadline; phased loop OFF
#
# Set after cell 9 on purpose: cell 9 asserts `inference` is not yet imported before it
# applies the graft's LOCAL_ANALYZER_* overrides, and nothing in inference/avo reads this
# flag at import time -- AvoSettings.from_env() is called per game inside the solver
# (_make_analyzer) and once at run start (_write_effective_flags), both after this cell.
os.environ["ARC3_AVO_PHASED_LOOP"] = "0"

import inference.avo.settings as _avo_settings_mod
import inference.avo.prompts as _avo_prompts_mod

# The module must come from the mounted AVO bundle, not from anything else on sys.path.
for _m in (_avo_settings_mod, _avo_prompts_mod):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)

# Read back through the solver's own call, not through os.environ.
_avo = _avo_settings_mod.AvoSettings.from_env()
assert _avo.phased_loop is False, _avo
# Everything else is the bundle's shipped default: one variable, and this proves it.
assert (_avo.stagnation_turns, _avo.unrewarded_turns, _avo.escalation_interventions) == (3, 12, 3), _avo
assert _avo.exploit_deadline == 0.6, _avo
assert (_avo.max_facts, _avo.max_rules, _avo.max_failures) == (24, 16, 16), _avo

# INSPECT is the directive that tells the model not to act. Recorded either way so the
# log says which arm ran without needing the kernel metadata.
print("AVO_ARM " + json.dumps(_avo.as_dict(), sort_keys=True), flush=True)
print(f"AVO_ARM phased_loop={_avo.phased_loop} inspect_directive_active={_avo.phased_loop} "
      f"module={Path(_avo_settings_mod.__file__).parent}", flush=True)


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise â€” an interactive "Save & Run" â€” play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple([
    "tn36-ef4dde99",
    "lf52-271a04aa",
    "cn04-2fe56bfb",
    "bp35-0a0ad940",
    "wa30-ee6fef47",
    "lp85-305b61c3",
    "r11l-495a7899",
    "tu93-0768757b",
    "sp80-589a99af",
    "m0r0-492f87ba",
    "vc33-5430563c",
    "ar25-0c556536",
    "ka59-38d34dbb",
    "sc25-635fd71a",
    "sk48-d8078629",
    "dc22-fdcac232",
    "cd82-fb555c5d",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ls20-9607627b",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "su15-1944f8ab",
    "tr87-cd924810"
])

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path(_COMP_DIR) / "environment_files")   # thui-animfast: resolved in cell 5
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError('The offline public game list contains duplicate IDs.')
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or extra:
        raise RuntimeError(
            f'Offline public game set changed; missing={missing}, extra={extra}.'
        )
    bm.games = [offline_by_id[game_id] for game_id in PUBLIC_GAME_IDS]
    if len(bm.games) != 25:
        raise RuntimeError(f'Expected 25 public games, got {len(bm.games)}.')
    print(f'PUBLIC25_SELECTION games={len(bm.games)} passes=1', flush=True)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if budget <= 600.0:
    raise RuntimeError(f'Notebook budget is too small for the teardown reserve: {budget}.')
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
    seconds=budget - 600.0
)

# Start recovery only after setup readiness and all run gates pass.
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / 'serving_setup.py')
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

# Play the benchmark; watchdog stop and teardown run even if it raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    bm._save_json()
    if not TRUE_SUBMISSION:
        # Kaggle Save & Run expects this valid placeholder after an offline run.
        # A real competition rerun uses the live gateway and never enters this branch.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

        # Check terminal coverage, then call the frozen scorer once.
        # This path does not render HTML.
        public_runs = list(bm.game_runs)
        public_run_ids = [run.game_id for run in public_runs]
        if len(public_runs) != 25 or public_run_ids != list(PUBLIC_GAME_IDS):
            raise RuntimeError(
                f'Public run coverage changed: count={len(public_runs)} ids={public_run_ids}.'
            )
        unfinished = [
            (run.game_id, run.state, run.final_score)
            for run in public_runs
            if run.state not in {'won', 'gave_up', 'cancelled'}
            or run.final_score is None
        ]
        if unfinished:
            raise RuntimeError(f'Public runs did not finalize cleanly: {unfinished}.')
        crashed = [run.game_id for run in public_runs if run.state == 'crashed']
        if crashed:
            raise RuntimeError(f'Public runs crashed: {crashed}.')
        total_actions = sum(len(run.history) for run in public_runs)
        if total_actions <= 0:
            raise RuntimeError('Public runs produced no actions.')

        from inference.tools.eval import evaluate_runs, save_score_file

        score_summary = evaluate_runs([WORKING_DIR])
        score_path = save_score_file(
            score_summary,
            run_dirs=[WORKING_DIR],
            output_path=WORKING_DIR / "score.json",
        )
        if Path(score_path) != WORKING_DIR / 'score.json' or not Path(score_path).is_file():
            raise RuntimeError(f'Frozen scorer did not write score.json: {score_path}.')
        print(
            f'PUBLIC25_AUDIT runs=25 actions={total_actions} score_path={score_path}',
            flush=True,
        )
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            print(f"taaf.kaggle: teardown command: {command}", flush=True)
            subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=30.0,
            )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
# Minimal diagnostics are enabled; skip post-run HTML rendering.
